In [1]:
# import
from pathlib import Path
import re

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap  # umap-learn

# paths
DATA_ROOT   = Path.home() / "vambersky_t/Data"
EMBED_ROOT  = DATA_ROOT / "embeddings"
WINDOWS_DIR = DATA_ROOT / "extracted_windows"

# configuration
SEQ_LEN  = 10_000
BIN_SIZE = 50
N_BINS   = SEQ_LEN // BIN_SIZE
EMBED_DIM   = 4096     # blocks.28.mlp.l3 hidden size


RANDOM_SEED = 42
N_PCA_COMPONENTS = 50   # fit PCA on full 4096-d, keep 50 for downstream use
N_UMAP_COMPONENTS = 2

# style
sns.set_theme(style="ticks", context="paper", font_scale=1.2)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})


/home/jovyan/envs/evo2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_filename(npz_path: Path) -> dict:
    """
    Extract accession, TF, and biosample from filename.
    Expected format: ACCESSION__TF__BIOSAMPLE.embeddings.npz
    """
    stem = npz_path.name.split(".embeddings")[0]
    accession, tf, biosample = stem.split("__")
    return {"accession": accession, "tf": tf, "biosample": biosample}

In [3]:
npz_files = sorted(EMBED_ROOT.glob("*/*.embeddings.npz"))

if not npz_files:
    raise FileNotFoundError(f"No embedding files found under {EMBED_ROOT}. "
                            "Run 02_embeddings.ipynb first.")

print(f"Found {len(npz_files)} embedding file(s):")
for f in npz_files:
    print(f"  {f.parent.name}/{f.name}")



Found 3 embedding file(s):
  CTCF/ENCFF017XLW__CTCF__GM12878.embeddings.npz
  MYC/ENCFF700CXD__MYC__H1.embeddings.npz
  MYC/ENCFF765CKK__MYC__GM12878.embeddings.npz


In [4]:
# Cell 3 — Load embeddings and assemble metadata

embeddings_list = []   # list of (n_peaks, 200, 4096) float16 arrays
meta_rows = []         # one dict per file, expanded to per-peak rows below

for npz_path in npz_files:
    info = parse_filename(npz_path)
    parquet_path = npz_path.with_suffix("").with_suffix("").with_name(
        npz_path.name.split(".embeddings")[0] + ".peak_ids.parquet"
    )

    if not parquet_path.exists():
        print(f"  WARNING: sidecar missing for {npz_path.name}, skipping")
        continue

    data = np.load(npz_path)
    emb  = data["embeddings"]   # (n_peaks, 200, 4096) float16
    ids  = pl.read_parquet(parquet_path)

    assert emb.shape[0] == len(ids), (
        f"Shape mismatch in {npz_path.name}: "
        f"{emb.shape[0]} embeddings vs {len(ids)} peak IDs"
    )
    assert emb.shape[1:] == (N_BINS, EMBED_DIM), (
        f"Unexpected embedding shape in {npz_path.name}: {emb.shape}"
    )

    embeddings_list.append(emb)

    # tag each peak with file-level metadata
    n = len(ids)
    meta_rows.append(
        ids.with_columns([
            pl.lit(info["accession"]).alias("accession"),
            pl.lit(info["tf"]).alias("tf"),
            pl.lit(info["biosample"]).alias("biosample"),
            pl.lit(npz_path.parent.name + "/" + npz_path.name).alias("bed_file"),
        ])
    )

    print(f"  Loaded {n:>6} peaks  {info['tf']:<6}  {info['biosample']:<12}  "
          f"{emb.nbytes / 1e9:.2f} GB")

# ── Concatenate ───────────────────────────────────────────────────────────────
meta = pl.concat(meta_rows)
meta_pd = meta.to_pandas()

print(f"\nTotal peaks loaded: {len(meta)}")
print(f"TFs present:        {meta['tf'].unique().to_list()}")
print(f"Biosamples present: {meta['biosample'].unique().to_list()}")
print(f"Files loaded:       {meta['bed_file'].n_unique()}")

  Loaded   9966 peaks  CTCF    GM12878       16.33 GB
  Loaded   4957 peaks  MYC     H1            8.12 GB
  Loaded   2216 peaks  MYC     GM12878       3.63 GB

Total peaks loaded: 17139
TFs present:        ['CTCF', 'MYC']
Biosamples present: ['H1', 'GM12878']
Files loaded:       3


In [5]:
def compute_gc_bins(sequence: str, n_bins: int = N_BINS, bin_size: int = BIN_SIZE) -> np.ndarray:
    """
    Compute GC content for each non-overlapping 50 bp bin of a 10 kb sequence.
    Returns a (200,) float32 array. Ns are excluded from the denominator —
    GC is computed as G+C / (A+C+G+T) within each bin, ignoring N.
    """
    gc = np.empty(n_bins, dtype=np.float32)
    for i in range(n_bins):
        bin_seq = sequence[i * bin_size : (i + 1) * bin_size]
        g = bin_seq.count("G")
        c = bin_seq.count("C")
        a = bin_seq.count("A")
        t = bin_seq.count("T")
        total = g + c + a + t
        gc[i] = (g + c) / total if total > 0 else np.nan
    return gc

def load_gc_for_file(accession: str, tf: str, biosample: str) -> np.ndarray:
    """
    Load the windows CSV for one BED file and compute per-bin GC content
    for all peaks. Returns (n_peaks, 200) float32.
    Peaks are returned in the same order as the CSV rows.
    """
    csv_path = WINDOWS_DIR / tf / f"{accession}__{tf}__{biosample}.full_peaks.csv.gz"
    if not csv_path.exists():
        raise FileNotFoundError(f"Windows file not found: {csv_path}")

    df = pl.read_csv(csv_path)
    sequences = df["sequence"].to_list()

    gc_matrix = np.stack([compute_gc_bins(seq, n_bins=N_BINS, bin_size=BIN_SIZE) for seq in sequences])
    return gc_matrix, df["peak_id"].to_list()

In [6]:
# verify

gc, ids = load_gc_for_file("ENCFF765CKK", "MYC", "GM12878")
print(gc.shape, gc.dtype)
print(f"NaN bins: {np.isnan(gc).sum()}")
print(f"GC range: {np.nanmin(gc):.3f} – {np.nanmax(gc):.3f}")

(2216, 200) float32
NaN bins: 1953
GC range: 0.000 – 1.000


In [7]:

gc, ids = load_gc_for_file("ENCFF765CKK", "MYC", "GM12878")

# how many peaks have at least one NaN bin?
peaks_with_nan = np.isnan(gc).any(axis=1).sum()
print(f"Peaks with ≥1 NaN bin: {peaks_with_nan} / {gc.shape[0]}")

# where in the 200-bin positional axis do NaNs occur?
nan_by_bin = np.isnan(gc).sum(axis=0)
print(f"Bins with any NaN: {(nan_by_bin > 0).sum()}")
print(f"Max NaNs in a single bin position: {nan_by_bin.max()}")

# are NaNs concentrated at the edges?
print(f"\nNaN counts, first 5 bins:  {nan_by_bin[:5]}")
print(f"NaN counts, last 5 bins:   {nan_by_bin[-5:]}")
print(f"NaN counts, middle 5 bins: {nan_by_bin[97:102]}")

Peaks with ≥1 NaN bin: 13 / 2216
Bins with any NaN: 200
Max NaNs in a single bin position: 11

NaN counts, first 5 bins:  [10 10 10 10 10]
NaN counts, last 5 bins:   [10 10 10 10 10]
NaN counts, middle 5 bins: [9 9 9 9 9]


In [8]:
# align by peak_id - we don't know how if the LOOP! preserved the order, 

def load_and_align(accession: str, tf: str, biosample: str) -> dict:
    """
    Load embeddings and GC content for one BED file, align by peak_id,
    and drop any peak containing at least one all-N bin (NaN in GC matrix).
    
    Returns a dict with keys:
        embeddings  : (n_peaks, 200, 4096) float16
        gc          : (n_peaks, 200) float32
        meta        : polars DataFrame with peak_id, chr, start, end, center
        accession   : str
        tf          : str
        biosample   : str
    """
    # ── Load embeddings + sidecar ─────────────────────────────────────────────
    npz_path     = EMBED_ROOT / tf / f"{accession}__{tf}__{biosample}.embeddings.npz"
    parquet_path = EMBED_ROOT / tf / f"{accession}__{tf}__{biosample}.peak_ids.parquet"

    data = np.load(npz_path, mmap_mode="r")
    emb  = data["embeddings"]                  # (n_peaks, 200, 4096) float16
    ids  = pl.read_parquet(parquet_path)       # peak_id, chr, start, end, center

    # ── Load GC ───────────────────────────────────────────────────────────────
    gc_matrix, gc_ids = load_gc_for_file(accession, tf, biosample)
    gc_df = pl.DataFrame({"peak_id": gc_ids})

    # ── Align by peak_id ──────────────────────────────────────────────────────
    # Build index maps: peak_id -> row position in each array
    emb_index = {pid: i for i, pid in enumerate(ids["peak_id"].to_list())}
    gc_index  = {pid: i for i, pid in enumerate(gc_ids)}

    # Intersection: peaks present in both arrays
    common_ids = sorted(set(emb_index) & set(gc_index))

    if len(common_ids) != len(emb_index):
        print(f"  WARNING: {len(emb_index) - len(common_ids)} peaks in embeddings "
              f"have no matching GC entry")
    if len(common_ids) != len(gc_index):
        print(f"  WARNING: {len(gc_index) - len(common_ids)} peaks in GC matrix "
              f"have no matching embedding")

    # Reindex both arrays to the common ordered peak set
    emb_rows = [emb_index[pid] for pid in common_ids]
    gc_rows  = [gc_index[pid]  for pid in common_ids]

    emb_aligned = emb[emb_rows]           # (n_common, 200, 4096)
    gc_aligned  = gc_matrix[gc_rows]      # (n_common, 200)

    # ── Drop NaN peaks ────────────────────────────────────────────────────────
    # A peak is dropped if any of its 200 bins is entirely N-masked
    nan_mask  = np.isnan(gc_aligned).any(axis=1)   # (n_common,) bool
    keep_mask = ~nan_mask

    n_dropped = nan_mask.sum()
    if n_dropped > 0:
        print(f"  Dropping {n_dropped} peak(s) with ≥1 all-N bin "
              f"({n_dropped / len(common_ids) * 100:.2f}%)")

    emb_clean = emb_aligned[keep_mask]    # (n_kept, 200, 4096)
    gc_clean  = gc_aligned[keep_mask]     # (n_kept, 200)
    ids_clean = [pid for pid, keep in zip(common_ids, keep_mask) if keep]

    # ── Rebuild metadata DataFrame ────────────────────────────────────────────
    meta_clean = (
        ids
        .filter(pl.col("peak_id").is_in(ids_clean))
        .with_columns([
            pl.lit(accession).alias("accession"),
            pl.lit(tf).alias("tf"),
            pl.lit(biosample).alias("biosample"),
        ])
    )

    print(f"  {accession}  {tf:<6}  {biosample:<12}  "
          f"{emb_clean.shape[0]} peaks retained")

    return {
        "embeddings": emb_clean,
        "gc":         gc_clean,
        "meta":       meta_clean,
        "accession":  accession,
        "tf":         tf,
        "biosample":  biosample,
    }

In [ ]:
result1 = load_and_align("ENCFF765CKK", "MYC", "GM12878")
print(result1["embeddings"].shape)
print(result1["gc"].shape)
print(result1["meta"].shape)

In [ ]:
result2 = load_and_align("ENCFF700CXD", "MYC", "H1")
print(result2["embeddings"].shape)
print(result2["gc"].shape)
print(result2["meta"].shape)